In [88]:
import json
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

In [89]:
WINDOW = 223
NUM_FEATURES = 18
WINDOW_STEP = 20
STEP = 20 # WINDOW // 4

CLOSE_POINTS_THRESHOLD = 60

In [90]:
current_dir = Path.cwd()
division_data_dir = current_dir.parent.parent / "data" / "division"
test_data_dir = division_data_dir / "test"
division_models_dir = current_dir.parent.parent / "models" / "division"

# Form random test stream and gesture targets

In [91]:
stream_ranges = pd.read_csv(division_data_dir / "stream_ranges.csv")
first_windows = set(stream_ranges["first_window_id"].values)

first_windows_test = []

for file in os.listdir(test_data_dir):
    if file.endswith(".csv"):
        window_id = int(file.split(".")[0])
        if window_id in first_windows:
            first_windows_test.append(window_id)
            
first_windows_test.sort()
first_window_id = random.choice(first_windows_test)
stream_id, last_window_id = stream_ranges.loc[
    stream_ranges["first_window_id"] == first_window_id, ["stream_id", "last_window_id"]
].values[0]

stream_windows_test = []

for file in os.listdir(test_data_dir):
    if file.endswith(".csv"):
        window_id = int(file.split(".")[0])
        if first_window_id <= window_id <= last_window_id:
            df = pd.read_csv(test_data_dir / file)
            stream_windows_test.append(
                (window_id, df.values.astype(np.float32))
            )
            
stream_windows_test.sort(key=lambda x: x[0])
stream_windows_test = [item[1] for item in stream_windows_test]

for i in range(len(stream_windows_test) - 1):
    stream_windows_test[i] = stream_windows_test[i][:WINDOW_STEP]
    
reconstructed_stream = np.concatenate(stream_windows_test, axis=0)

stream_gestures = pd.read_csv(division_data_dir / "stream_gestures.csv")
boundaries_list = stream_gestures.loc[
    stream_gestures["stream_id"] == stream_id, ["start_sample", "end_sample"]
]

# Inference

In [92]:
scaler = joblib.load(division_models_dir / "scaler.pkl")
has_start_model = tf.keras.models.load_model(division_models_dir / "has_start.keras")
has_end_model = tf.keras.models.load_model(division_models_dir / "has_end.keras")
start_norm_model = tf.keras.models.load_model(division_models_dir / "start_norm.keras")
end_norm_model = tf.keras.models.load_model(division_models_dir / "end_norm.keras")

with open(division_models_dir / "thresholds.json", "r") as f:
    thresholds = json.load(f)


detected_starts = []
detected_ends = []

N_samples = len(reconstructed_stream)

for left in range(0, N_samples - WINDOW + 1, STEP):
    right = left + WINDOW
    window_data = reconstructed_stream[left:right]
    
    window_2d = window_data.reshape(-1, NUM_FEATURES)
    window_scaled = scaler.transform(window_2d).reshape(1, WINDOW, NUM_FEATURES)
    
    has_start_prob = has_start_model.predict(window_scaled, verbose=0)
    has_end_prob = has_end_model.predict(window_scaled, verbose=0)
    
    has_start_prob = has_start_prob[0][0]
    has_end_prob = has_end_prob[0][0]
    
    if has_start_prob >= thresholds["has_start"]:
        start_norm = start_norm_model.predict(window_scaled, verbose=0)
        
        local_idx = int(start_norm[0][0] * (WINDOW - 1))
        absolute_idx = left + local_idx
        detected_starts.append(absolute_idx)
        
    if has_end_prob >= thresholds["has_end"]:
        end_norm = end_norm_model.predict(window_scaled, verbose=0)
        
        local_idx = int(end_norm[0][0] * (WINDOW - 1))
        absolute_idx = left + local_idx
        detected_ends.append(absolute_idx)


def group_and_average_points(points, threshold):
    """Groups neighboring points from different windows that point to the same event"""
    if not points:
        return []
    points = sorted(points)
    groups = [[points[0]]]
    
    for p in points[1:]:
        if p - groups[-1][-1] <= threshold:
            groups[-1].append(p)
        else:
            groups.append([p])
            
    return [int(np.mean(g)) for g in groups]

final_starts = group_and_average_points(detected_starts, CLOSE_POINTS_THRESHOLD)
final_ends = group_and_average_points(detected_ends, CLOSE_POINTS_THRESHOLD)

print(f"Filtered Absolute Starts: {final_starts}")
print(f"Filtered Absolute Ends: {final_ends}")
print(reconstructed_stream.shape)
boundaries_list

Filtered Absolute Starts: [68, 365, 615]
Filtered Absolute Ends: [10, 331, 584, 721]
(803, 18)


,start_sample,end_sample
338,66,330
339,364,583
340,617,778
